# YOLO Object Detection Notebook

本 Notebook 将原来的 `.py` 检测程序转换为 Jupyter Notebook 格式。

支持三种输入方式：
- **Webcam**：实时摄像头检测
- **Image**：单张图片检测并保存结果
- **Video**：视频逐帧检测并保存结果

代码尽量保持原程序的行为，同时补充了详细中文注释，方便后续学习和修改。

In [2]:
# 1. 安装 ultralytics（包含 YOLOv8 API）
!pip install -U ultralytics
# Install PyTorch with CUDA
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.4/65.4 kB 4.4 MB/s eta 0:00:00
Looking in indexes: https://download.pytorch.org/whl/cu121


## 1. 导入依赖

`ultralytics` 用于加载 YOLO 模型并执行目标检测；`opencv-python`（`cv2`）负责读取、显示和保存图片/视频；`pathlib` 和 `os` 用于路径处理；这里不再需要 `argparse` 和 `sys`，因为 Notebook 通常直接在配置单元中修改参数。

In [3]:
from ultralytics import YOLO
import cv2
import os
from pathlib import Path

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.


## 2. 配置参数

这是 Notebook 中最重要的配置区域。运行前只需要修改这里即可。

- `SOURCE` 可以写成 `"webcam"`，也可以填写图片或视频路径。
- `CONF_THRESHOLD` 是置信度阈值。原程序默认是 `0.05`，这里保持不变。
- `WEIGHTS` 是训练好的 YOLO 权重文件路径。
- `OUTPUT_DIR` 是检测结果保存目录。
- `CAM_INDEX` 是摄像头编号，通常电脑自带摄像头为 `0`。
- `SAVE_WEBCAM` 决定是否同时保存摄像头检测视频。

In [5]:
# =========================
# 用户配置区域
# =========================

# 检测来源：
# 1. "webcam"       -> 使用摄像头
# 2. 图片路径         -> 检测单张图片
# 3. 视频路径         -> 检测视频
SOURCE = "images/1.jpg"

# YOLO 置信度阈值。
# 数值越低，会保留更多低置信度检测结果，但误检可能增加。
CONF_THRESHOLD = 0.05

# 训练好的 YOLO 权重文件。
WEIGHTS = "runs/detect/train2/weights/best.pt"

# 检测结果保存目录。
OUTPUT_DIR = "runs/detect/predict"

# 摄像头编号。通常：0 = 默认摄像头，1 = 第二个摄像头。
CAM_INDEX = 0

# 是否保存 webcam 检测结果视频。
SAVE_WEBCAM = False

## 3. 辅助函数：创建目录

检测结果保存之前，需要确保输出目录存在。

`os.makedirs(..., exist_ok=True)` 在目录已经存在时不会报错，因此可以安全地重复调用。

In [6]:
def ensure_dir(path):
    """确保指定目录存在；如果不存在则创建。"""
    os.makedirs(path, exist_ok=True)

## 4. 判断输入文件类型

根据文件扩展名判断输入是图片还是视频。

使用 `Path(path).suffix.lower()` 可以统一处理大写扩展名，例如 `.JPG` 会被转换成 `.jpg`。

In [7]:
def is_image_file(path):
    """判断给定路径是否属于常见图片格式。"""
    ext = Path(path).suffix.lower()
    return ext in [
        '.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.tif'
    ]


def is_video_file(path):
    """判断给定路径是否属于常见视频格式。"""
    ext = Path(path).suffix.lower()
    return ext in [
        '.mp4', '.avi', '.mov', '.mkv', '.webm', '.mpeg'
    ]

## 5. 保存检测后的图片

`results[0].plot()` 会返回已经绘制检测框、类别名称和置信度等信息的图像数组。

`cv2.imwrite()` 再把这个 BGR 格式的 NumPy 数组保存到磁盘。

In [8]:
def save_image(annotated_frame, out_path):
    """保存 YOLO 绘制后的图片，并检查保存是否成功。"""
    ok = cv2.imwrite(out_path, annotated_frame)

    if not ok:
        raise RuntimeError(f"cv2.imwrite failed for {out_path}")

## 6. 图片检测函数

处理流程：
1. 使用 YOLO 模型读取图片并执行检测。
2. 使用 `results[0].plot()` 绘制检测结果。
3. 创建输出目录。
4. 使用原图片文件名保存检测后的图片。

In [9]:
def detect_image_file(model, image_path, output_dir, conf_threshold=0.05):
    """检测单张图片，并将带检测框的结果保存到输出目录。"""

    # 对图片执行 YOLO 推理。
    # conf 用于控制最低置信度阈值。
    results = model.predict(image_path, conf=conf_threshold)

    # 取得第一张输入图片对应的检测结果，并绘制检测框。
    annotated_frame = results[0].plot()

    # 确保输出目录存在。
    ensure_dir(output_dir)

    # 保留原始图片文件名。
    out_filename = Path(image_path).name
    out_path = os.path.join(output_dir, out_filename)

    # 保存图片。
    save_image(annotated_frame, out_path)

    print(f"Annotated image saved to {out_path}")

## 7. 视频检测函数

视频不能像单张图片一样一次性处理，因此这里采用逐帧处理：

`VideoCapture` → 读取 frame → YOLO 检测 → 绘制结果 → `VideoWriter` 写入输出视频。

这样可以保持原视频的宽度、高度和帧率信息。

In [10]:
def detect_video_file(
    model,
    video_path,
    output_dir,
    conf_threshold=0.05,
    fps_override=None,
):
    """检测视频文件，并将带检测结果的视频保存到输出目录。"""

    # 打开输入视频。
    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        raise RuntimeError(f"Could not open video file {video_path}")

    # 获取视频的基本参数。
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # 如果用户提供 fps_override，则优先使用它；
    # 否则使用原视频 FPS；如果原视频没有有效 FPS，则使用 25 FPS。
    fps = fps_override or cap.get(cv2.CAP_PROP_FPS) or 25.0

    # 创建输出目录。
    ensure_dir(output_dir)

    # 生成输出文件名，例如：video.mp4 -> video_annotated.mp4。
    out_filename = Path(video_path).stem + "_annotated.mp4"
    out_path = os.path.join(output_dir, out_filename)

    # 使用 mp4v 编码器生成 MP4 文件。
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(
        out_path,
        fourcc,
        fps,
        (width, height),
    )

    # 用于记录已经处理了多少帧。
    frame_idx = 0

    while True:
        # 读取下一帧。
        ret, frame = cap.read()

        # ret=False 表示视频读取结束或读取失败。
        if not ret:
            break

        # Ultralytics 可以直接接收 OpenCV 的 NumPy frame。
        results = model.predict(frame, conf=conf_threshold)

        # 在当前帧上绘制检测结果。
        annotated = results[0].plot()

        # 如果绘制结果为空，则跳过当前帧。
        if annotated is None:
            print(
                f"Warning: annotated frame is None at index {frame_idx}"
            )
            continue

        # 将检测后的帧写入输出视频。
        writer.write(annotated)
        frame_idx += 1

    # 释放视频资源。
    cap.release()
    writer.release()

    print(f"Annotated video saved to {out_path}")

## 8. Webcam 实时检测

摄像头模式和视频模式的主要区别是：摄像头需要实时显示画面。

`cv2.imshow()` 用于显示窗口；按键盘 **q** 可以退出检测循环。

如果 `SAVE_WEBCAM=True`，程序还会使用 `VideoWriter` 将检测后的画面保存成 MP4。

In [11]:
def detect_webcam(
    model,
    conf_threshold=0.05,
    cam_index=0,
    output_dir=None,
    save_video=False,
):
    """使用摄像头进行实时 YOLO 检测，可选保存检测视频。"""

    # 打开指定编号的摄像头。
    cap = cv2.VideoCapture(cam_index)

    if not cap.isOpened():
        print("Error: Could not open webcam")
        return

    writer = None
    out_path = None

    # 如果用户要求保存 webcam 视频，并且指定了输出目录，
    # 则初始化 VideoWriter。
    if save_video and output_dir:
        ensure_dir(output_dir)

        out_path = os.path.join(
            output_dir,
            "webcam_annotated.mp4",
        )

        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps = cap.get(cv2.CAP_PROP_FPS) or 20.0

        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        writer = cv2.VideoWriter(
            out_path,
            fourcc,
            fps,
            (width, height),
        )

    print("Starting webcam detection... Press 'q' to quit")

    while True:
        # 从摄像头读取一帧。
        ret, frame = cap.read()

        if not ret:
            print("Error: Could not read frame")
            break

        # 对当前帧执行 YOLO 检测。
        results = model.predict(frame, conf=conf_threshold)

        # 绘制检测结果。
        annotated_frame = results[0].plot()

        # 如果绘制失败，则退回使用原始 frame，避免窗口没有画面。
        if annotated_frame is None:
            annotated_frame = frame

        # 显示实时检测结果。
        cv2.imshow("YOLO Detection", annotated_frame)

        # 如果启用了视频保存，则写入当前检测帧。
        if writer:
            writer.write(annotated_frame)

        # waitKey(1) 等待键盘输入约 1 ms。
        # 按下 q 时退出循环。
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    # 释放摄像头资源。
    cap.release()

    # 如果创建了视频写入器，也要释放它。
    if writer:
        writer.release()
        print(f"Webcam annotated video saved to {out_path}")

    # 关闭 OpenCV 创建的所有窗口。
    cv2.destroyAllWindows()

## 9. 检查模型权重并加载 YOLO 模型

在运行检测之前，先确认 `WEIGHTS` 指向的 `.pt` 文件存在。

然后使用 `YOLO(WEIGHTS)` 加载训练好的模型。

In [12]:
# 检查权重文件是否存在。
if not os.path.exists(WEIGHTS):
    raise FileNotFoundError(
        f"Weights file not found: {WEIGHTS}"
    )

# 加载训练好的 YOLO 模型。
model = YOLO(WEIGHTS)

print(f"Model loaded successfully: {WEIGHTS}")

FileNotFoundError: Weights file not found: runs/detect/train2/weights/best.pt

## 10. 执行检测

这里对应原 `.py` 文件中的 `main()`。

Notebook 不再使用 `argparse`，而是直接读取前面的配置变量，然后自动判断：

- `SOURCE == "webcam"` → 摄像头检测
- 图片扩展名 → 图片检测
- 视频扩展名 → 视频检测
- 无法识别扩展名 → 先尝试图片，再尝试视频

In [ ]:
source = SOURCE
conf = CONF_THRESHOLD
output_dir = OUTPUT_DIR

# =========================
# Webcam
# =========================
if source.lower() == "webcam":
    detect_webcam(
        model,
        conf_threshold=conf,
        cam_index=CAM_INDEX,
        output_dir=output_dir,
        save_video=SAVE_WEBCAM,
    )

# =========================
# 图片 / 视频文件
# =========================
else:
    # 检查输入文件是否存在。
    if not os.path.exists(source):
        raise FileNotFoundError(
            f"Source file does not exist: {source}"
        )

    # 根据文件扩展名判断类型。
    if is_image_file(source):
        try:
            detect_image_file(
                model,
                source,
                output_dir,
                conf_threshold=conf,
            )
        except Exception as error:
            print(f"Error while processing image: {error}")
            raise

    elif is_video_file(source):
        try:
            detect_video_file(
                model,
                source,
                output_dir,
                conf_threshold=conf,
            )
        except Exception as error:
            print(f"Error while processing video: {error}")
            raise

    # 如果扩展名没有在预定义列表中，则按照原程序的思路：
    # 先尝试图片，失败后再尝试视频。
    else:
        try:
            detect_image_file(
                model,
                source,
                output_dir,
                conf_threshold=conf,
            )
        except Exception:
            try:
                detect_video_file(
                    model,
                    source,
                    output_dir,
                    conf_threshold=conf,
                )
            except Exception as error:
                print(f"Error: could not process file {source}: {error}")
                raise

## 11. 使用说明

### Webcam
把配置改成：

```python
SOURCE = "webcam"
CAM_INDEX = 0
SAVE_WEBCAM = False
```

运行检测单元后，按 **q** 退出。

### 图片
例如：

```python
SOURCE = "data/test.jpg"
```

结果会保存到：

```text
runs/detect/predict/test.jpg
```

### 视频
例如：

```python
SOURCE = "data/test.mp4"
```

结果会保存为类似：

```text
runs/detect/predict/test_annotated.mp4
```

### 重要提醒
如果你在 Jupyter Notebook / JupyterLab / VS Code Notebook 中运行 Webcam，OpenCV 的 `cv2.imshow()` 在不同环境中的行为可能不同。若窗口无法正常显示，通常需要根据你使用的 Notebook 环境调整显示方式。